In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- 1. Data Loading ---
# Load the datasets
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
sample_submission_df = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

# --- 2. Text Preprocessing ---
# A simple function to clean the text data
def clean_text(text):
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and special characters, keeping only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

# Apply the cleaning function to the 'text' column of both dataframes
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# --- 3. Vectorization (Text to Numbers) ---
# Initialize the TF-IDF Vectorizer
# max_features limits the vocabulary size for efficiency
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit the vectorizer on the training data and transform both train and test data
X_train = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])
X_test = tfidf_vectorizer.transform(test_df['cleaned_text'])

# The target variable
y_train = train_df['target']

# --- 4. Model Training ---
# Initialize the Logistic Regression model
model = LogisticRegression(C=1.0, random_state=42)

# Train the model
model.fit(X_train, y_train)

# --- 5. Prediction and Submission ---
# Make predictions on the test data
test_predictions = model.predict(X_test)

# Create the submission file
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'target': test_predictions
})

# Save the submission file
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print("A preview of the submission file:")
print(submission_df.head())